# 🧠 AI Logic Engine: Knowledge Ingestion (Firestore)

### ⚠️ အသုံးပြုပုံ အဆင့်ဆင့် (Instructions):
1. **Runtime Type:** `T4 GPU` ဖြစ်နေရပါမယ်။ (Runtime > Change runtime type > T4 GPU)
2. **Step 2:** Firebase Console မှ ဒေါင်းလုဒ်လုပ်ထားသော `serviceAccountKey.json` ကို Upload တင်ပါ။
3. **Step 5:** `Run All` လုပ်ထားလျှင်ပင် Step 5 ရောက်သည့်အခါ **Target Goal** (ဘယ်နှစ်ခု သွင်းမလဲ) ကို Manual ရိုက်ထည့်ပေးရပါမည်။

---

In [ ]:
# Step 1: Install & Imports
!pip install -q firebase-admin transformers accelerate bitsandbytes torch tqdm

import os, json, re, torch, time
from tqdm.auto import tqdm
import firebase_admin
from firebase_admin import credentials, firestore
from google.colab import files
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("✅ Setup Complete.")

In [ ]:
# Step 2: Firestore Connection
DATABASE_ID = 'ai-studio-09d97652-b1c3-4b1a-b63e-31d8a38be4c2'

if not firebase_admin._apps:
    print("Please upload your serviceAccountKey.json file:")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    cred = credentials.Certificate(filename)
    firebase_admin.initialize_app(cred)

db = firestore.client(database_id=DATABASE_ID)
print(f"✅ Memory Connected to Firestore: {DATABASE_ID}")

In [ ]:
# Step 3: Core Logic Utilities
STATE_FILE = "state_v3.json"

def clean_id(text):
    if not text: return ""
    text = text.lower().strip()
    return re.sub(r'[^a-z0-9]+', '_', text).strip('_')

def normalize_verb(v):
    is_a_patterns = ['is_a', 'is a', 'is type of', 'belongs to', 'category of']
    if any(p == v.strip().lower() for p in is_a_patterns): return 'is_a'
    return v.strip().lower().replace(' ', '_')

def save_progress(count):
    with open(STATE_FILE, "w") as f: json.dump({"count": count}, f)

def load_progress():
    if os.path.exists(STATE_FILE):
        try:
            with open(STATE_FILE, "r") as f: return json.load(f).get("count", 0)
        except: pass
    return 0

print("✅ Utilities Armed.")

In [ ]:
# Step 4: Load AI Model
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
print("⏳ Loading Model (This may take a few minutes)... ")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
model.eval()
print("✅ AI Logic Model is ready.")

In [ ]:
# @title 🚀 Step 5: Start Ingestion Loop (Robust & Debug Enabled)
try:
    TARGET_INPUT = input("Enter Target Goal (Facts count, e.g. 1000): ")
    TARGET_GOAL = int(TARGET_INPUT) if TARGET_INPUT else 1000
    print(f"🎯 Target set to: {TARGET_GOAL}")
except:
    TARGET_GOAL = 1000
    print("Invalid input. Defaulting to 1000 facts.")

CATEGORIES = ["Biology", "Physics", "History", "Neuroscience", "Economics", "Technology", "Environment"]
current_total = load_progress()
pbar = tqdm(initial=current_total, total=TARGET_GOAL, desc="Ingesting Symbols")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

while current_total < TARGET_GOAL:
    try:
        cat = CATEGORIES[current_total % len(CATEGORIES)]
        print(f"\n⏳ Generating Knowledge for: {cat}...")
        
        prompt = f"List exactly 12 definitive facts about {cat}. Format: Subject|Verb|Object|Sentence. No intro, no numbering, strictly pipe-separated."
        messages = [
            {"role": "system", "content": "Output ONLY pipe-separated data format: S|V|O|Sentence."},
            {"role": "user", "content": prompt}
        ]
        
        input_txt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(input_txt, return_tensors="pt", padding=True).to(model.device)
        
        with torch.no_grad():
            out = model.generate(
                inputs.input_ids, 
                attention_mask=inputs.attention_mask,
                max_new_tokens=512, 
                do_sample=True, 
                temperature=0.8,
                pad_token_id=tokenizer.pad_token_id
            )
        
        response = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        
        if not response.strip():
            print("⚠️ AI returned empty result. Skipping...")
            continue

        batch = db.batch()
        added_count = 0
        lines = response.strip().split('\n')
        
        for line in lines:
            if '|' not in line: continue
            parts = [x.strip() for x in line.split('|')]
            if len(parts) >= 3:
                s, v, o = parts[0], normalize_verb(parts[1]), parts[2]
                sid, oid = clean_id(s), clean_id(o)
                
                if sid and oid and sid != oid:
                    node_ref = db.collection('nodes').document(sid)
                    data = {'name': s, 'updatedAt': firestore.SERVER_TIMESTAMP}
                    if v == 'is_a':
                        data['groups'] = firestore.ArrayUnion([oid])
                    else:
                        sent = parts[3] if len(parts) > 3 else f"{s} {v.replace('_',' ')} {o}."
                        data['relations'] = firestore.ArrayUnion([{
                            'verb': v, 'targetId': oid, 'sentence': sent, 'weight': 100
                        }])
                    batch.set(node_ref, data, merge=True)
                    added_count += 1
        
        if added_count > 0:
            print(f"✅ Syncing {added_count} symbols to database...")
            batch.commit()
            current_total += added_count
            pbar.update(added_count)
            save_progress(current_total)
        else:
            print("❌ Format Mismatch. No facts were extracted from this AI generation.")
            
        if current_total % 25 == 0: torch.cuda.empty_cache()
        
    except Exception as e:
        print(f"⚠️ Error: {e}. Retrying in 5s...")
        time.sleep(5)
    except KeyboardInterrupt:
        print("\n🛑 Stopping ingestion process.")
        break

pbar.close()
print(f"✅ Ingestion Cycle Finished. Total Knowledge Base Size: {current_total}")